# 11 — Wildfire demo → GeoAI triage → Sentinel-2 STAC

**[🚀 Launch this notebook live](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=11_wildfire_stac_triage.ipynb)**

This lab starts with the actual polygons in GeoLibre’s [US Wildfire Perimeters 2025](https://share.geolibre.app/giswqs/us-wildfire-perimeters-2025) demo. It then:

1. engineers size, season, perimeter, and compactness features;
2. uses Isolation Forest to surface unusual fire profiles;
3. chooses one large unusual fire as an investigation target;
4. searches Element 84’s Earth Search STAC API for nearby Sentinel-2 Level-2A scenes; and
5. ranks candidate acquisitions and returns the perimeter, footprints, and best visual COG to GeoLibre.

> This is an imagery-discovery exercise, not operational fire intelligence. Use current incident-authority products for safety or emergency decisions.

In [ ]:
import sys
if sys.platform == 'emscripten':
    import micropip
    await micropip.install(['geolibre==3.0.0', 'pyodide-http'])
    import pyodide_http
    pyodide_http.patch_all()

import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from IPython.display import IFrame, Markdown, display
from geolibre_lite import LiteMap as Map

assert Map.__module__ == 'geolibre_lite'

In [ ]:
DEMO = 'https://share.geolibre.app/giswqs/us-wildfire-perimeters-2025'
PROJECT_JSON = DEMO + '.geolibre.json'
display(IFrame(DEMO + '?embed=1', width='100%', height=620))

## Extract and enrich the perimeter layer

The GeoLibre project is a portable JSON document. No scraping is required: its first layer is an inline GeoJSON FeatureCollection containing the authored perimeter data and popup attributes.

In [ ]:
response = requests.get(PROJECT_JSON, timeout=90)
response.raise_for_status()
project = response.json()
perimeters_fc = next(layer['geojson'] for layer in project['layers'] if 'perimeter' in layer['name'].lower())
fires = gpd.GeoDataFrame.from_features(perimeters_fc['features'], crs='EPSG:4326')
fires['discovered'] = pd.to_datetime(fires['discovered'])
fires['day_of_year'] = fires['discovered'].dt.dayofyear

metric = fires.to_crs(5070)
fires['area_km2_geometry'] = metric.area / 1e6
fires['perimeter_km'] = metric.length / 1e3
fires['compactness'] = (4 * np.pi * metric.area / metric.length.pow(2)).clip(0, 1)

print(project['metadata']['description'])
fires[['fire', 'state', 'cause', 'acres', 'discovered', 'area_km2_geometry', 'compactness']].head()

## Find unusual fire profiles

Isolation Forest combines reported acreage, discovery date, perimeter length, and shape compactness. A high score means that a fire is unusual relative to the other large 2025 fires in this demo; it does **not** measure severity, damage, or response priority.

In [ ]:
model_columns = ['acres', 'day_of_year', 'perimeter_km', 'compactness']
X = fires[model_columns].copy()
X['acres'] = np.log1p(X['acres'])
X['perimeter_km'] = np.log1p(X['perimeter_km'])
Xs = StandardScaler().fit_transform(X)

detector = IsolationForest(n_estimators=300, contamination=0.08, random_state=42)
fires['unusual_profile'] = detector.fit_predict(Xs) == -1
fires['anomaly_score'] = -detector.score_samples(Xs)

unusual = fires[fires['unusual_profile']].sort_values(['acres', 'anomaly_score'], ascending=False)
display(unusual[['fire', 'state', 'cause', 'acres', 'discovered', 'compactness', 'anomaly_score']].head(12))
target = unusual.iloc[0] if len(unusual) else fires.nlargest(1, 'acres').iloc[0]
print(f"Investigation target: {target['fire']} ({target['state']}), {target['acres']:,.0f} acres")

## Search Earth Search STAC

The search uses the target perimeter’s bounding box and a 60-day window centered on discovery. The `fields` extension keeps the response small while retaining geometry, cloud cover, acquisition time, platform, and the public visual COG. Open the [Earth Search catalog](https://earth-search.aws.element84.com/v1) or a [STAC Browser view](https://radiantearth.github.io/stac-browser/#/external/earth-search.aws.element84.com/v1) to continue interactively.

In [ ]:
minx, miny, maxx, maxy = target.geometry.bounds
pad = 0.15
bbox = [minx - pad, miny - pad, maxx + pad, maxy + pad]
start = (target['discovered'] - pd.Timedelta(days=30)).strftime('%Y-%m-%dT00:00:00Z')
end = (target['discovered'] + pd.Timedelta(days=30)).strftime('%Y-%m-%dT23:59:59Z')

payload = {
    'collections': ['sentinel-2-c1-l2a'],
    'bbox': bbox,
    'datetime': f'{start}/{end}',
    'limit': 60,
    'query': {'eo:cloud_cover': {'lt': 80}},
    'fields': {
        'include': ['id', 'bbox', 'geometry', 'properties.datetime', 'properties.eo:cloud_cover', 'properties.platform', 'assets.visual'],
        'exclude': ['links'],
    },
}
stac_response = requests.post('https://earth-search.aws.element84.com/v1/search', json=payload, timeout=90)
stac_response.raise_for_status()
items = stac_response.json()['features']
if not items:
    raise RuntimeError('No Sentinel-2 scenes matched. Increase the date window or cloud threshold.')
print(f'{len(items)} Sentinel-2 candidate scenes returned')

## Rank acquisition candidates

Cloud cover and temporal distance from discovery are normalized into a simple, auditable score; lower is better. K-Means adds an exploratory acquisition-profile label. This ranking only triages metadata—actual usefulness depends on smoke, local cloud over the fire, sensor coverage, and the analysis question.

In [ ]:
scene_rows, scene_geometries = [], []
for item in items:
    props = item['properties']
    acquired = pd.to_datetime(props['datetime']).tz_localize(None)
    scene_rows.append({
        'item_id': item['id'],
        'acquired': acquired,
        'days_from_discovery': (acquired.normalize() - target['discovered']).days,
        'cloud_cover': float(props['eo:cloud_cover']) if props.get('eo:cloud_cover') is not None else 100.0,
        'platform': props.get('platform', 'unknown'),
        'visual_cog': item.get('assets', {}).get('visual', {}).get('href'),
    })
    scene_geometries.append(shape(item['geometry']))
scenes = gpd.GeoDataFrame(scene_rows, geometry=scene_geometries, crs='EPSG:4326')
scenes['triage_score'] = 0.65 * (scenes['cloud_cover'] / 100) + 0.35 * (scenes['days_from_discovery'].abs().clip(upper=30) / 30)

if len(scenes) >= 3:
    cluster_X = StandardScaler().fit_transform(scenes[['cloud_cover', 'days_from_discovery']])
    scenes['acquisition_profile'] = KMeans(n_clusters=3, n_init=20, random_state=42).fit_predict(cluster_X)
else:
    scenes['acquisition_profile'] = 0

scenes = scenes.sort_values('triage_score').reset_index(drop=True)
best_scene = scenes.iloc[0]
display(scenes[['item_id', 'acquired', 'days_from_discovery', 'cloud_cover', 'acquisition_profile', 'triage_score']].head(12))

## Inspect the handoff in GeoLibre

The demo perimeters are colored by anomaly score, the selected fire is outlined, the STAC footprints are overlaid, and the highest-ranked scene’s visual COG is streamed directly from Earth Search. Toggle layers and verify spatial coverage before attempting burn-severity or change analysis.

In [ ]:
center = target.geometry.centroid
m = Map(center=(center.x, center.y), zoom=7, height='760px')
if best_scene['visual_cog']:
    m.add_cog(best_scene['visual_cog'], name=f"Best candidate — {best_scene['acquired'].date()}", bands=[1, 2, 3], rescale=[0, 255])
fires_map = fires.copy()
fires_map['discovered'] = fires_map['discovered'].dt.strftime('%Y-%m-%d')
m.add_choropleth(fires_map, column='anomaly_score', name='2025 fire anomaly score', class_count=7, colormap='magma', fillOpacity=0.45, strokeWidth=1)
target_feature = {'type': 'Feature', 'properties': {'fire': target['fire'], 'acres': float(target['acres'])}, 'geometry': target.geometry.__geo_interface__}
m.add_geojson({'type': 'FeatureCollection', 'features': [target_feature]}, name='Investigation target', fillOpacity=0.05, strokeColor='#00ffff', strokeWidth=4)
scenes_map = scenes.head(12).copy()
scenes_map['acquired'] = scenes_map['acquired'].dt.strftime('%Y-%m-%dT%H:%M:%S')
m.add_geojson(scenes_map, name='Top Sentinel-2 footprints', fillColor='#38bdf8', fillOpacity=0.08, strokeColor='#0284c7', strokeWidth=2)
m

## Continue toward a full GeoAI workflow

- Search separately for one clear pre-fire and one clear post-fire acquisition.
- Use the red, NIR, and SWIR assets advertised by the same STAC Items to calculate NBR or dNBR.
- Clip reads to the perimeter plus a buffer instead of downloading complete scenes.
- In full CPython/GPU infrastructure, hand the paired chips to a segmentation or change-detection model and record model/version provenance.

## Data & software citations

- [GeoLibre US Wildfire Perimeters 2025 demo](https://share.geolibre.app/giswqs/us-wildfire-perimeters-2025) and [project JSON](https://share.geolibre.app/giswqs/us-wildfire-perimeters-2025.geolibre.json). The demo identifies NIFC WFIGS Interagency Fire Perimeters as its source.
- [Element 84 Earth Search](https://github.com/Element84/earth-search) STAC API and Copernicus Sentinel-2 Collection 1 Level-2A COGs. Check the STAC Collection metadata for current licensing and attribution.
- [STAC specification](https://stacspec.org/) and [STAC API specification](https://github.com/radiantearth/stac-api-spec).
- GeoLibre: https://geolibre.app/ — MIT-licensed software.
- scikit-learn supplies the exploratory Isolation Forest and K-Means models.